# Customer Lifetime Value (CLTV) Prediction

## Project Overview

This project develops a machine learning model to predict Customer Lifetime Value (CLTV) using customer-level insurance data.

The objective is to identify patterns in customer characteristics and build a regression model capable of estimating the potential lifetime value of customers.

This project was developed as part of a **Customer Lifetime Value Prediction Hackathon**.


## Business Problem

Businesses need to understand which customers are likely to generate higher long-term value.

Accurate CLTV prediction can help businesses:

- Identify high-value customers
- Improve customer retention
- Prioritize marketing campaigns
- Allocate marketing resources
- Support customer segmentation
- Develop personalized customer strategies


## Project Objectives

- Understand the customer dataset
- Clean and preprocess the data
- Handle missing values
- Perform exploratory data analysis
- Engineer useful features
- Transform the target variable
- Encode categorical variables
- Train a machine learning regression model
- Evaluate model performance using R²
- Analyze feature importance
- Generate CLTV predictions for unseen customers
- Derive business-oriented insights


## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Load Dataset

The original competition notebook used Kaggle-specific paths. For this portfolio repository, the dataset is expected to be stored locally in the `data/` directory.

Expected structure:

```text
data/
├── train.csv
└── test.csv
```

If your actual filenames are different, update the paths below.


In [ ]:
TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Training shape:", train.shape)
print("Test shape:", test.shape)


## 3. Dataset Overview

In [ ]:
display(train.head())
display(test.head())


In [ ]:
print("Training information:")
train.info()

print("\nTest information:")
test.info()


In [ ]:
print("Training descriptive statistics:")
display(train.describe(include="all").T)


In [ ]:
print("Missing values in training data:")
display(train.isnull().sum().sort_values(ascending=False))

print("\nMissing values in test data:")
display(test.isnull().sum().sort_values(ascending=False))


## 4. Exploratory Data Analysis

Exploratory analysis helps identify distributions, unusual values, and relationships that may be useful for CLTV prediction.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(train["cltv"].dropna(), bins=30)
plt.title("Distribution of Customer Lifetime Value")
plt.xlabel("CLTV")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()


In [ ]:
if "income" in train.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(train["income"].dropna(), bins=30)
    plt.title("Distribution of Customer Income")
    plt.xlabel("Income")
    plt.ylabel("Number of Customers")
    plt.tight_layout()
    plt.show()


In [ ]:
if "claim_amount" in train.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(train["claim_amount"].dropna(), bins=30)
    plt.title("Distribution of Claim Amount")
    plt.xlabel("Claim Amount")
    plt.ylabel("Number of Customers")
    plt.tight_layout()
    plt.show()


In [ ]:
if "num_policies" in train.columns:
    plt.figure(figsize=(8, 5))
    train["num_policies"].value_counts().sort_index().plot(kind="bar")
    plt.title("Number of Policies Held by Customers")
    plt.xlabel("Number of Policies")
    plt.ylabel("Number of Customers")
    plt.tight_layout()
    plt.show()


In [ ]:
numeric_cols_eda = train.select_dtypes(include=np.number).columns

if len(numeric_cols_eda) > 1:
    corr = train[numeric_cols_eda].corr()
    plt.figure(figsize=(10, 7))
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()


## 5. Data Quality Checks

The dataset is checked for:

- Missing values
- Duplicate records
- Data types
- Unexpected values
- Numerical and categorical features


In [ ]:
print("Duplicate rows in training data:", train.duplicated().sum())
print("Duplicate rows in test data:", test.duplicated().sum())

print("\nTraining data types:")
display(train.dtypes)


## 6. Data Preprocessing

The preprocessing workflow includes:

1. Cleaning column names
2. Separating the customer ID
3. Converting numerical variables
4. Handling missing values
5. Separating the target variable


In [ ]:
train.columns = train.columns.str.strip().str.lower()
test.columns = test.columns.str.strip().str.lower()

print("Training columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())


In [ ]:
test_ids = test["id"].copy()

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

print("ID removed from model features.")


In [ ]:
num_cols = ["income", "claim_amount", "vintage", "num_policies"]

for col in num_cols:
    if col in train.columns:
        train[col] = pd.to_numeric(train[col], errors="coerce")
    if col in test.columns:
        test[col] = pd.to_numeric(test[col], errors="coerce")

existing_num_cols = [col for col in num_cols if col in train.columns]

train[existing_num_cols] = train[existing_num_cols].fillna(0)
test[existing_num_cols] = test[existing_num_cols].fillna(0)

print("Missing numerical values handled.")


## 7. Feature Engineering

Additional customer-level features are created to help the model capture relationships between income, claims, policies, and customer vintage.


In [ ]:
def create_features(df):
    df = df.copy()

    df["income_to_claim"] = df["income"] / (df["claim_amount"] + 1)
    df["claim_to_income"] = df["claim_amount"] / (df["income"] + 1)
    df["premium_per_policy"] = df["income"] / (df["num_policies"] + 1)
    df["policy_per_year"] = df["num_policies"] / (df["vintage"] + 1)

    df["income_log"] = np.log1p(df["income"].clip(lower=0))
    df["claim_log"] = np.log1p(df["claim_amount"].clip(lower=0))
    df["policies_log"] = np.log1p(df["num_policies"].clip(lower=0))

    df["income_per_year"] = df["income"] / (df["vintage"] + 1)
    df["claim_per_policy"] = df["claim_amount"] / (df["num_policies"] + 1)
    df["risk_score"] = df["claim_amount"] / (df["income"] + 1)

    return df

train = create_features(train)
test = create_features(test)

print("Feature engineering completed.")
print("Training shape after feature engineering:", train.shape)


## 8. Target Variable Preparation

The target variable is `cltv`.

Because CLTV can be highly skewed, a log transformation is applied using `log1p()`. Predictions are later converted back to the original CLTV scale using `expm1()`.


In [ ]:
y = train["cltv"].copy()
X = train.drop(columns=["cltv"]).copy()

y_log = np.log1p(y)

print("Original CLTV summary:")
display(y.describe())

print("\nLog-transformed CLTV summary:")
display(y_log.describe())


## 9. Categorical Encoding

Categorical variables are converted into numerical columns using one-hot encoding.

The training and test datasets are aligned so that both contain the same feature columns.


In [ ]:
X = pd.get_dummies(X, drop_first=False)
test_encoded = pd.get_dummies(test, drop_first=False)

X, test_encoded = X.align(
    test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

X = X.fillna(0)
test_encoded = test_encoded.fillna(0)

print("Encoded training shape:", X.shape)
print("Encoded test shape:", test_encoded.shape)


## 10. Train-Validation Split

The data is divided into 80% training data and 20% validation data.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_log,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])


## 11. Model Training

### LightGBM Regressor

LightGBM is a gradient boosting framework designed for efficient learning on structured and tabular datasets.

It was selected because it can model nonlinear relationships and interactions between features effectively.


In [ ]:
model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed.")


## 12. Model Evaluation

The primary evaluation metric is the **R² score**.


In [ ]:
val_preds = model.predict(X_val)

validation_r2 = r2_score(y_val, val_preds)

print(f"Validation R² Score: {validation_r2:.4f}")


## 13. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance_df.head(15))


In [ ]:
top_n = 15
plot_df = importance_df.head(top_n).sort_values("Importance")

plt.figure(figsize=(9, 6))
plt.barh(plot_df["Feature"], plot_df["Importance"])
plt.title("Top 15 Feature Importances - LightGBM")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()

os.makedirs("../images", exist_ok=True)
plt.savefig("../images/feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()


## 14. Final Model Training

After validation, the final model is trained using the complete training dataset before generating predictions for the unseen test dataset.


In [ ]:
final_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    random_state=42
)

final_model.fit(X, y_log)

print("Final model trained on the complete training dataset.")


## 15. Test Prediction

Predictions are generated for the unseen test dataset and converted back to the original CLTV scale.


In [ ]:
test_preds_log = final_model.predict(test_encoded)
test_preds = np.expm1(test_preds_log)

print("Prediction generation completed.")
print("Number of predictions:", len(test_preds))


## 16. Prediction Output

The final prediction file contains:

| Column | Description |
|---|---|
| `id` | Customer identifier |
| `cltv` | Predicted Customer Lifetime Value |


In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "cltv": test_preds
})

display(submission.head(10))


In [ ]:
os.makedirs("../outputs", exist_ok=True)

submission.to_csv(
    "../outputs/submission.csv",
    index=False
)

print("Submission file saved to ../outputs/submission.csv")


## 17. Prediction Summary

In [ ]:
display(submission["cltv"].describe())


## 18. Business Insights

The model can support customer-level value analysis.

Potential insights include:

1. Customers with higher predicted CLTV can be considered potential high-value customer segments.
2. Feature importance can help identify customer characteristics that contribute strongly to model predictions.
3. CLTV predictions can support customer segmentation and marketing prioritization.
4. Predicted customer value can be combined with other customer behavior information to develop more targeted strategies.

These insights should be interpreted together with the actual feature-importance results and customer prediction distribution.


## 19. Business Recommendations

Based on the purpose of the CLTV prediction model, businesses can consider:

- Prioritizing high-CLTV customers for retention campaigns.
- Developing personalized offers for valuable customer segments.
- Using predicted CLTV to support marketing resource allocation.
- Designing customer segments based on expected lifetime value.
- Monitoring customer value over time.
- Retraining the model as new customer data becomes available.


## 20. Conclusion

This project demonstrates an end-to-end machine learning workflow for Customer Lifetime Value prediction.

The workflow includes data preprocessing, exploratory data analysis, feature engineering, categorical encoding, target transformation, LightGBM regression, model evaluation, feature importance analysis, test prediction, and submission file generation.

The resulting CLTV predictions can potentially support customer segmentation, retention strategies, marketing prioritization, and data-driven customer relationship decisions.


## 21. Portfolio Project Summary

**Project:** Customer Lifetime Value (CLTV) Prediction

**Model:** LightGBM Regressor

**Evaluation Metric:** R² Score

**Validation R²:** Approximately 0.3134 in the original validation run

**Key Skills Demonstrated:**

- Python
- Pandas
- NumPy
- Exploratory Data Analysis
- Data Preprocessing
- Feature Engineering
- Machine Learning
- LightGBM
- Model Evaluation
- Feature Importance
- Business Insights
